# RECAST Pipeline
End-to-end: raw data → cleaning → augmentation → LoRA fine-tuning → cross-validation.

## Config

In [ ]:
from pathlib import Path

RAW_DATA       = Path("datasets/RECAST-30K.json")
CLEAN_DATA     = Path("datasets/recast_30k_clean.jsonl")
AUGMENTED_DATA = Path("datasets/recast_30k_augmented.jsonl")

## Step 1 — Cleaning

In [ ]:
from src.crllm.dataset.preprocess.preprocess import run_pipeline

clean_stats = run_pipeline(
    input_path=RAW_DATA,
    output_path=CLEAN_DATA,
    min_length=15,
    dedup_threshold=0.85,
    imbalance_threshold=0.5,
)
print(clean_stats)

## Step 2 — Augmentation

In [ ]:
from src.crllm.dataset.augmentation.augment import run_augmentation

aug_stats = run_augmentation(
    input_path=CLEAN_DATA,
    output_path=AUGMENTED_DATA,
    seed=42,
)
print(aug_stats)

## Step 3 — LoRA Fine-Tuning

**TODO:** Refactor `lora_finetuning_basic/lora_base.py` into a callable API, then wire it up here.

Needed: `run_lora_finetune(dataset_path, output_path, **config)` that accepts at minimum:
- `dataset_path` — path to the augmented JSONL
- `output_path` — where to save the adapter weights
- `model_name`, `lora_rank`, `lora_alpha`, `learning_rate`, `num_epochs`

```python
# from lora_finetuning_basic.lora_base import run_lora_finetune
#
# lora_stats = run_lora_finetune(
#     dataset_path=AUGMENTED_DATA,
#     output_path=Path("outputs/lora_adapter"),
#     model_name="meta-llama/Llama-3.2-1B-Instruct",
#     lora_rank=8,
#     learning_rate=1e-4,
#     num_epochs=1,
# )
# print(lora_stats)
```

## Step 4 — Cross-Validation

**TODO:** Refactor `Cross_validation/cross_validation_kfold.ipynb` into a callable API, then wire it up here.

Needed: `run_cross_validation(dataset_path, lora_adapter_path, **config)` that accepts at minimum:
- `dataset_path` — path to the evaluation dataset
- `lora_adapter_path` — path to the trained LoRA adapter from Step 3
- `k_folds`, `samples_per_fold`

```python
# from Cross_validation.cross_validation_kfold import run_cross_validation
#
# cv_results = run_cross_validation(
#     dataset_path=AUGMENTED_DATA,
#     lora_adapter_path=Path("outputs/lora_adapter"),
#     k_folds=5,
#     samples_per_fold=200,
# )
# print(cv_results)
```